<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0;">Lab 02: Create Your First LangGraph Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">Build the travel concierge as a declarative state graph with nodes, edges, and compiled execution</p>
</div>

**What We Learn:** How to build a travel concierge as a declarative state graph — defining nodes for LLM calls, edges for control flow, and compiling into a runnable agent served via Foundry.

---


## 🧳 The Contoso Travel Story

Building the same concierge but as a graph — each step in the conversation flow is a visible node.

- **The Problem:** With MAF's BaseAgent, the control flow is hidden inside Python methods. For complex workflows with branching and loops, it's hard to reason about the execution path.
- **This Lab Solves:** Defining the concierge as a `StateGraph` where the flow is explicit: START → llm_call → END. Easy to visualize, test, and extend.


## 1. Setup

Load environment variables and create the Foundry project client.

---


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.projects import AIProjectClient

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
print(f"✅ Connected to Foundry project")
print(f"   Model: {model_name}")

## 2. Initialize the LLM

Create an <code>AzureChatOpenAI</code> instance using Azure AD authentication. This is the LangChain-compatible LLM that our graph nodes will call.

---


In [ ]:
from langchain_openai import AzureChatOpenAI

# Get the token provider for Azure AD auth
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

# Extract the base endpoint from the project endpoint
base_endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"].split("/api/projects")[0]

llm = AzureChatOpenAI(
    azure_deployment=model_name,
    azure_endpoint=base_endpoint,
    azure_ad_token_provider=token_provider,
    api_version="2025-01-01",
)
print(f"✅ LLM initialized: {model_name}")

## 3. Define the State and LLM Node

We use <code>MessagesState</code> to track conversation history and define an <code>llm_call</code> node that processes messages through the LLM.

---


In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import SystemMessage

CONCIERGE_INSTRUCTIONS = """You are the Contoso Travel concierge agent. You help customers plan trips by:
- Recommending destinations based on their preferences
- Suggesting travel dates and itineraries
- Providing budget estimates for flights, hotels, and car rentals
- Answering questions about travel requirements (visas, vaccines, etc.)

Be friendly, knowledgeable, and proactive. Ask clarifying questions when needed.
Always respond in a helpful, concise manner."""


def llm_call(state: MessagesState):
    """Process conversation through the LLM with travel instructions."""
    messages = [SystemMessage(content=CONCIERGE_INSTRUCTIONS)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


print("✅ LLM node defined: llm_call")

## 4. Build and Compile the Graph

Create the <code>StateGraph</code>, add the node, wire the edges, and compile it into a runnable agent.

---


In [ ]:
# Define the graph: START → llm_call → END
graph = StateGraph(MessagesState)
graph.add_node("llm_call", llm_call)
graph.add_edge(START, "llm_call")
graph.add_edge("llm_call", END)

# Compile into a runnable agent
agent = graph.compile()
print("✅ Graph compiled: START → llm_call → END")

## 5. Test the Agent

Invoke the compiled graph with a travel planning query and inspect the response.

---


In [ ]:
from langchain_core.messages import HumanMessage

# Invoke the compiled graph
result = agent.invoke(
    {"messages": [HumanMessage(content="I want to plan a trip from Seattle to Paris for two weeks in July.")]}
)

print("🧳 Concierge Response:")
print("-" * 50)
print(result["messages"][-1].content)

## 6. Visualize the Graph

LangGraph can render the graph structure as ASCII art, making the control flow visible.

---


In [ ]:
# Display the graph structure
agent.get_graph().print_ascii()

## 7. Serve with Foundry Hosting Adapter

In production, the hosting adapter wraps the compiled graph for the Foundry Responses API — the same pattern used by MAF.

---


In [ ]:
# Production: from_langgraph(agent).run() starts HTTP server on port 8088
# Same pattern as MAF: hosting adapter wraps your graph for the Responses API
print("📋 Production deployment:")
print("   from azure.ai.agentserver.langgraph import from_langgraph")
print("   from_langgraph(agent).run()")

## 8. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2e8b57; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>✅ What We Accomplished</b><br/>Built a travel concierge as a LangGraph state graph with an LLM node, compiled it, tested it with a real query, and visualized the graph structure.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>💡 Key Takeaway</b><br/>A LangGraph agent is a <b>compiled state graph</b>. Define nodes (functions), wire edges (control flow), compile, and invoke. The graph makes the execution path explicit and inspectable.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #ff9800; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>➡️ Next:</b> In <a href="lab-03-tools.ipynb">Lab 03</a>, we'll add tool-calling nodes with conditional routing — creating the classic ReAct loop where the LLM can search flights, hotels, and car rentals.
</div>


In [ ]:
# Cleanup: close clients
project_client.close()
credential.close()
print("✅ Clients closed.")